# 第一階段:載入並理解 NFL 資料

**最終目標**:做一個「傳球空間分析」工具,給教練或球探使用。

**本階段只做一件事**:把資料載進來、看懂每個檔案在講什麼、確認它們怎麼互相連接。

我們**不會**在這個階段建儀表板,也**不會**在分析資料前就先下結論。

---

### 先認識一點美式足球的基本詞彙(白話版)

- **play(一次進攻)**:球隊從把球發出(snap)到這次攻防結束,算一個 play。一場比賽有上百個 play。
- **offense(進攻方)/ defense(防守方)**:拿球、想前進得分的是進攻方;阻止對方的是防守方。
- **down(檔數)**:進攻方有 4 次機會(4 downs)推進 10 碼。推進到就重新拿到 4 次機會。
- **yards to go(還差幾碼)**:距離「推進 10 碼拿到新的一組 down」還差多少碼。
- **line of scrimmage(攻防線)**:這個 play 開始時球所在的那條橫線。
- **pass(傳球)/ 本專案重點**:四分衛(QB)把球丟給隊友。我們最後要分析「傳球時場上有多少空間」。
- **tracking data(追蹤資料)**:場上每位球員每 0.1 秒的座標位置,像 GPS 一樣。

## 步驟 0:載入工具與設定路徑

- **輸入**:無
- **輸出**:`pandas`(讀 CSV 的工具)、以及資料夾路徑設定
- **目的**:準備好後面每一步都會用到的工具與路徑。`pandas` 是 Python 處理表格資料最常用的套件,通常簡寫成 `pd`。

In [1]:
import sys
from pathlib import Path

import pandas as pd

# 讓筆記本找得到上一層資料夾裡的 config.py
sys.path.append(str(Path.cwd().parent))
from config import DATA_DIR, SAMPLE_GAME_ID

print("pandas 版本:", pd.__version__)
print("原始資料夾:", DATA_DIR)
print("資料夾存在嗎?", DATA_DIR.exists())
print("示範用的 gameId:", SAMPLE_GAME_ID)

pandas 版本: 2.2.3
原始資料夾: /Users/jingleliao/Downloads/nfl-big-data-bowl-regional-event-data-main/data
資料夾存在嗎? True
示範用的 gameId: 2021090900


## 步驟 1:載入 games.csv(比賽清單)

- **輸入**:`games.csv`
- **輸出**:一個 DataFrame(表格),變數名 `games`
- **目的**:這個檔是「比賽層級」的資料。

**一列代表什麼?** 一列 = 一場比賽。

**重要欄位**:
- `gameId`:比賽編號(唯一)。這是把所有檔案串起來的**主鑰匙**。
- `season` / `week`:球季 / 第幾週。
- `homeTeamAbbr` / `visitorTeamAbbr`:主隊 / 客隊的三字母代碼(例如 TB = Tampa Bay)。

In [2]:
games = pd.read_csv(DATA_DIR / "games.csv")

print("games 形狀 (列數, 欄數):", games.shape)
games.head()

games 形狀 (列數, 欄數): (122, 7)


,gameId,season,week,gameDate,gameTimeEastern,homeTeamAbbr,visitorTeamAbbr
0,2021090900,2021,1,09/09/2021,20:20:00,TB,DAL
1,2021091200,2021,1,09/12/2021,13:00:00,ATL,PHI
2,2021091201,2021,1,09/12/2021,13:00:00,BUF,PIT
3,2021091202,2021,1,09/12/2021,13:00:00,CAR,NYJ
4,2021091203,2021,1,09/12/2021,13:00:00,CIN,MIN


## 步驟 2:載入 plays.csv(每一次進攻)

- **輸入**:`plays.csv`
- **輸出**:DataFrame `plays`
- **目的**:這個檔是「play 層級」的資料,是我們之後分析的核心。

**一列代表什麼?** 一列 = 某場比賽裡的某一次進攻(play)。

**重要欄位**:
- `gameId` + `playId`:兩個合起來才能唯一指出一個 play(`playId` 在不同比賽間會重複)。
- `possessionTeam`:**進攻方**(拿球的隊)。
- `defensiveTeam`:**防守方**。
- `down` / `yardsToGo`:第幾檔 / 還差幾碼。
- `passResult`:傳球結果。`C`=傳球成功、`I`=沒接到、`S`=QB 被擒殺、`IN`=被抄截、`R`=QB 自己跑。
- `playDescription`:這次進攻的文字描述(人看得懂的說明)。

In [3]:
plays = pd.read_csv(DATA_DIR / "plays.csv")

print("plays 形狀 (列數, 欄數):", plays.shape)
plays[[
    "gameId", "playId", "possessionTeam", "defensiveTeam",
    "down", "yardsToGo", "passResult", "playDescription",
]].head()

plays 形狀 (列數, 欄數): (8557, 32)


,gameId,playId,possessionTeam,defensiveTeam,down,yardsToGo,passResult,playDescription
0,2021090900,97,TB,DAL,3,2,I,(13:33) (Shotgun) T.Brady pass incomplete deep...
1,2021090900,137,DAL,TB,1,10,C,(13:18) (Shotgun) D.Prescott pass deep left to...
2,2021090900,187,DAL,TB,2,6,C,(12:23) (Shotgun) D.Prescott pass short middle...
3,2021090900,282,DAL,TB,1,10,I,(9:56) D.Prescott pass incomplete deep left to...
4,2021090900,349,DAL,TB,3,15,I,(9:46) (Shotgun) D.Prescott pass incomplete sh...


## 步驟 3:載入 players.csv(球員名冊)

- **輸入**:`players.csv`
- **輸出**:DataFrame `players`
- **目的**:把球員編號對應到人名與位置。

**一列代表什麼?** 一列 = 一位球員。

**重要欄位**:
- `nflId`:球員編號(唯一)。這是連接球員資料的**主鑰匙**。
- `displayName`:球員姓名。
- `officialPosition`:守備位置(例如 QB=四分衛、WR=外接員、CB=角衛)。

In [4]:
players = pd.read_csv(DATA_DIR / "players.csv")

print("players 形狀 (列數, 欄數):", players.shape)
players.head()

players 形狀 (列數, 欄數): (1679, 7)


,nflId,height,weight,birthDate,collegeName,officialPosition,displayName
0,25511,6-4,225,1977-08-03,Michigan,QB,Tom Brady
1,28963,6-5,240,1982-03-02,"Miami, O.",QB,Ben Roethlisberger
2,29550,6-4,328,1982-01-22,Arkansas,T,Jason Peters
3,29851,6-2,225,1983-12-02,California,QB,Aaron Rodgers
4,30078,6-2,228,1982-11-24,Harvard,QB,Ryan Fitzpatrick


## 步驟 4:載入 pffScoutingData.csv(球探評分)

- **輸入**:`pffScoutingData.csv`
- **輸出**:DataFrame `pff`
- **目的**:這是 PFF(Pro Football Focus)這家公司提供的「每個 play、每位球員」的球探標記。

**一列代表什麼?** 一列 = 某個 play 裡的某一位球員。

**重要欄位**:
- `gameId` + `playId` + `nflId`:三個合起來唯一指出「哪場、哪個 play、哪位球員」。
- `pff_role`:這位球員在這個 play 的角色,例如 `Pass`=傳球者、`Pass Route`=跑路線接球、`Pass Block`=保護 QB、`Pass Rush`=衝向 QB、`Coverage`=防守盯人/區域。
- `pff_positionLinedUp`:發球瞬間站的位置。

> 註:很多欄位值是 `NA`(空白),代表這個指標對該球員這個 play 不適用,很正常。

In [5]:
pff = pd.read_csv(DATA_DIR / "pffScoutingData.csv")

print("pff 形狀 (列數, 欄數):", pff.shape)
pff[["gameId", "playId", "nflId", "pff_role", "pff_positionLinedUp"]].head()

pff 形狀 (列數, 欄數): (188254, 15)


,gameId,playId,nflId,pff_role,pff_positionLinedUp
0,2021090900,97,25511,Pass,QB
1,2021090900,97,35481,Pass Route,TE-L
2,2021090900,97,35634,Pass Route,LWR
3,2021090900,97,39985,Pass Route,HB-R
4,2021090900,97,40151,Pass Block,C


## 步驟 5:載入「一場比賽」的 tracking 追蹤資料

- **輸入**:`tracking/tracking_<gameId>.csv`(只載入一場,檔案很大)
- **輸出**:DataFrame `tracking`
- **目的**:這是場上每位球員(與球)每 0.1 秒的座標,是未來做「空間分析」的原料。

**一列代表什麼?** 一列 = 某個 play、某一位球員(或球)、在某一個時間點(frame)的位置。

**重要欄位**:
- `gameId` + `playId` + `nflId` + `frameId`:定位到「哪場、哪個 play、誰、第幾個時間點」。`nflId` 空白代表那一列是**球**。
- `x`, `y`:球員在場上的座標(碼)。`x` 是長邊 0–120,`y` 是短邊 0–53.3。
- `s`, `a`, `dir`, `o`:速度、加速度、移動方向、面向。
- `event`:這一刻發生的事件標記,例如 `ball_snap`(發球)、`pass_forward`(往前傳)、`pass_outcome_caught`(接到)。大多數時間點是 `None`(沒有特別事件)。

In [6]:
tracking_file = DATA_DIR / "tracking" / f"tracking_{SAMPLE_GAME_ID}.csv"
print("要讀的檔案:", tracking_file.name)

tracking = pd.read_csv(tracking_file)

print("tracking 形狀 (列數, 欄數):", tracking.shape)
tracking.head()

要讀的檔案: tracking_2021090900.csv
tracking 形狀 (列數, 欄數): (92644, 16)


,gameId,playId,nflId,frameId,time,jerseyNumber,team,playDirection,x,y,s,a,dis,o,dir,event
0,2021090900,97,25511.0,1,2021-09-10T00:26:31Z,12.0,TB,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN
1,2021090900,97,25511.0,2,2021-09-10T00:26:31Z,12.0,TB,right,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN
2,2021090900,97,25511.0,3,2021-09-10T00:26:31Z,12.0,TB,right,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN
3,2021090900,97,25511.0,4,2021-09-10T00:26:31Z,12.0,TB,right,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN
4,2021090900,97,25511.0,5,2021-09-10T00:26:31Z,12.0,TB,right,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN


## 步驟 6:確認真實的季別、週次、事件標籤

- **輸入**:上面載入的 `games` 與 `tracking`
- **輸出**:幾個統計表
- **目的**:**不要憑印象,直接用資料本身驗證**我們手上的是哪一季、哪幾週、以及有哪些事件標籤。

In [7]:
print("=== 球季 (season) ===")
print(games["season"].value_counts())

print("\n=== 每週場數 (week) ===")
print(games["week"].value_counts().sort_index())

print("\n=== 總場數 ===")
print(len(games), "場比賽")

=== 球季 (season) ===
season
2021    122
Name: count, dtype: int64

=== 每週場數 (week) ===
week
1    16
2    16
3    16
4    16
5    16
6    14
7    13
8    15
Name: count, dtype: int64

=== 總場數 ===
122 場比賽


In [8]:
print("=== 這場比賽 tracking 裡出現的 event 標籤 ===")
# dropna=False 讓我們也看到被記成空值的筆數
print(tracking["event"].value_counts(dropna=False))

=== 這場比賽 tracking 裡出現的 event 標籤 ===
event
NaN                          85997
ball_snap                     2231
pass_forward                  2116
autoevent_passforward          851
autoevent_ballsnap             667
play_action                    345
line_set                       115
pass_arrived                    69
autoevent_passinterrupted       69
run                             46
fumble                          23
fumble_offense_recovered        23
qb_sack                         23
man_in_motion                   23
pass_outcome_caught             23
pass_outcome_incomplete         23
Name: count, dtype: int64


## 步驟 7:看懂檔案之間怎麼連接

- **目的**:理解「鑰匙欄位(key)」怎麼把四個檔案 + tracking 串在一起。

連接關係(由大到小):

```
games    ── gameId ──┐
                     ├── plays        (用 gameId + playId 定位一個 play)
                     ├── pff          (用 gameId + playId + nflId)
                     └── tracking     (用 gameId + playId + nflId + frameId)

players  ── nflId ───> 連到 pff / tracking,把編號換成人名與位置
```

下面示範:挑這場比賽的**第一個 play**,把它在各檔案裡的資料找出來。

In [9]:
# 1) 這場比賽的基本資訊
this_game = games[games["gameId"] == SAMPLE_GAME_ID]
print("這場比賽:")
print(this_game[["gameId", "season", "week", "homeTeamAbbr", "visitorTeamAbbr"]].to_string(index=False))

# 2) 挑這場比賽的第一個 play
game_plays = plays[plays["gameId"] == SAMPLE_GAME_ID]
sample_play_id = game_plays["playId"].iloc[0]
print("\n我們挑的 playId:", sample_play_id)

# 3) 看這個 play 的描述
one_play = game_plays[game_plays["playId"] == sample_play_id]
print("\n這個 play 的文字描述:")
print(one_play["playDescription"].iloc[0])

這場比賽:
    gameId  season  week homeTeamAbbr visitorTeamAbbr
2021090900    2021     1           TB             DAL

我們挑的 playId: 97

這個 play 的文字描述:
(13:33) (Shotgun) T.Brady pass incomplete deep right to C.Godwin.


In [10]:
# 4) 用 gameId + playId + nflId 把 pff 裡這個 play 的球員,接上 players 的人名
pff_this_play = pff[(pff["gameId"] == SAMPLE_GAME_ID) & (pff["playId"] == sample_play_id)]

# merge = 依照共同的 nflId 把兩張表合併(類似 Excel VLOOKUP)
pff_named = pff_this_play.merge(
    players[["nflId", "displayName", "officialPosition"]],
    on="nflId",
    how="left",
)

print("這個 play 參與的球員與角色:")
pff_named[["displayName", "officialPosition", "pff_role", "pff_positionLinedUp"]]

這個 play 參與的球員與角色:


,displayName,officialPosition,pff_role,pff_positionLinedUp
0,Tom Brady,QB,Pass,QB
1,Rob Gronkowski,TE,Pass Route,TE-L
2,Antonio Brown,WR,Pass Route,LWR
3,Giovani Bernard,RB,Pass Route,HB-R
4,Ryan Jensen,C,Pass Block,C
5,Mike Evans,WR,Pass Route,RWR
6,Demarcus Lawrence,DE,Pass Rush,LEO
7,Donovan Smith,T,Pass Block,LT
8,Randy Gregory,DE,Pass Rush,ROLB
9,Ali Marpet,G,Pass Block,LG


## 步驟 8:你的小練習 🏈

**任務**:在這場比賽裡,找出 `playId = 137` 這個 play 的**進攻方**與**防守方**。

**提示**:
- 進攻方欄位叫 `possessionTeam`,防守方欄位叫 `defensiveTeam`。
- 先用 `gameId` 和 `playId` 篩出那一列。

先在下面這個格子自己試試看,再往下看答案。

In [11]:
# 👉 在這裡寫你的程式(提示:仿照步驟 7 的篩選方式)



### 怎麼檢查你的答案

執行下面這格,它會直接印出標準答案。把你的結果跟它比對即可。

另外也教你一個「交叉驗證」小技巧:`playDescription` 文字描述裡通常會出現發動進攻那隊的球員名字,可以當作旁證。

In [12]:
answer = plays[(plays["gameId"] == SAMPLE_GAME_ID) & (plays["playId"] == 137)]

print("進攻方 (possessionTeam):", answer["possessionTeam"].iloc[0])
print("防守方 (defensiveTeam):", answer["defensiveTeam"].iloc[0])
print("\n旁證 —— 這個 play 的描述:")
print(answer["playDescription"].iloc[0])

進攻方 (possessionTeam): DAL
防守方 (defensiveTeam): TB

旁證 —— 這個 play 的描述:
(13:18) (Shotgun) D.Prescott pass deep left to A.Cooper pushed ob at DAL 30 for 28 yards (A.Winfield).


## 本階段完成!

你已經:
1. 載入了 5 個檔案(4 個主檔 + 1 場 tracking)。
2. 知道每個檔案「一列代表什麼」、重要欄位的意義。
3. 用資料本身驗證了球季(2021)、週次(1–8)、事件標籤。
4. 看懂了 `gameId` / `playId` / `nflId` / `frameId` 怎麼把檔案串起來。
5. 完成了一個找「進攻方 / 防守方」的小練習。

**下一階段預告**(等你確認理解後再開始):開始聚焦單一傳球 play,從 tracking 資料找出發球(ball_snap)到傳球(pass_forward)之間的畫面,為「傳球空間」打基礎。